In [139]:
import os
import ast
import fitz
import json
import torch
import base64
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from colpali_engine.models import ColQwen2, ColQwen2Processor

import psycopg2
from pgvector.psycopg2 import register_vector

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

In [140]:
with open("idf_map.json", "r") as f:
    idf_map = json.load(f)

In [189]:
idf_map

{'comprehensive': 3.1780538303479458,
 'version': 3.1780538303479458,
 'for': 0.0,
 'government': 0.2336148511815051,
 'suppliers': 0.2336148511815051,
 'procurement': 0.28768207245178085,
 'guide': 1.3862943611198906,
 'ethics': 2.4849066497880004,
 'rounds': 2.4849066497880004,
 'registering': 2.4849066497880004,
 'results': 2.4849066497880004,
 'products': 2.0794415416798357,
 'tender': 0.4054651081081644,
 'checklist': 2.4849066497880004,
 '16': 3.1780538303479458,
 'as': 0.28768207245178085,
 'reviewing': 2.0794415416798357,
 'or': 0.2336148511815051,
 'alignment': 2.4849066497880004,
 'standards': 1.3862943611198906,
 'feedback': 1.791759469228055,
 '5': 1.791759469228055,
 'site': 2.4849066497880004,
 'avoid': 2.0794415416798357,
 '21': 3.1780538303479458,
 'competitive': 1.3862943611198906,
 'and': 0.08701137698962969,
 '9': 2.0794415416798357,
 '7': 2.4849066497880004,
 '1': 1.2321436812926323,
 'you': 0.2336148511815051,
 '18': 3.1780538303479458,
 'useful': 0.875468737353899

In [2]:
pdf_path = "pdfs\supplier_guide_detailed.pdf"

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_26632\394254503.py:1: SyntaxWarning: invalid escape sequence '\s'
  pdf_path = "pdfs\supplier_guide_detailed.pdf"


In [3]:
load_dotenv()

True

In [4]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [6]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [7]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 4485.79it/s]
ColQwen2 LOAD REPORT from: vidore/colqwen2-v1.0
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
base_model.language_model.model.custom_text_proj.lora_B.default.weight | UNEXPECTED | 
base_model.language_model.model.custom_text_proj.lora_A.default.weight | UNEXPECTED | 
custom_text_proj.lora_A.default.weight                                 | MISSING    | 
custom_text_proj.lora_B.default.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [9]:
def get_coarse_embedding(embeddings_tensor):
    normalized = embeddings_tensor / embeddings_tensor.norm(dim=-1, keepdim=True)
    return normalized.mean(dim=1).squeeze().to(torch.float32).cpu() 

In [10]:
def embed_query(query, model, processor):
    inputs = processor.process_queries(queries=[query])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
        embeddings = embeddings.squeeze(0).to(torch.float32).cpu().numpy()
    return coarse_vector, embeddings


In [11]:
def numerical_expansion(query, llm):
    if not any(char.isdigit() or char == '$' for char in query):
        return query

    prompt = f"""
    Act as a search query optimizer for technical documents. 
    Rewrite the user query to maximize retrieval accuracy by following these rules:
    
    1. NUMERICAL BOUNDARIES: If a specific value or number is mentioned, include the 
       likely category or threshold it belongs to (e.g., if a user asks for 'age 5', 
       add 'early childhood' or 'pediatric').
    2. TECHNICAL SYNONYMS: Add common industry-standard synonyms for the actions described.
    3. NO CONVERSATION: Do not explain the change. Return only the keywords.
    
    User Query: {query}
    Optimized Query:"""
    
    return llm.invoke(prompt).content


In [144]:
def get_query_weights(query_tokens):
    weights = []
    for t in query_tokens:
        clean_t = t.replace('Ġ', '').lower().strip()
        w = idf_map.get(clean_t, 1.0)
        if any(char.isdigit() for char in clean_t) or '$' in clean_t:
            w *= 1.5
        weights.append(w)
    return np.array(weights, dtype=np.float32)

In [15]:
def coarse_search(coarse_vector, top_k = 20):
    try:
        with conn.cursor() as cur:
            cur.execute('''
                SELECT id FROM pages
                ORDER BY coarse_vector <=> %s::vector
                LIMIT %s

            ''', (coarse_vector, top_k))
            ids = [row[0] for row in cur.fetchall()]
            return ids
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [304]:
def precision_search_diverse_weighted(query, query_embeddings, ids, top_k=3):
    inputs = processor(text=[query], return_tensors="pt")
    input_ids = inputs["input_ids"][0]
    query_tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
    query_weights = get_query_weights(query_tokens)
    num_embeddings = query_embeddings.shape[0]
    num_tokens = len(query_weights)
    final_weights = np.ones(num_embeddings, dtype=np.float32)
    final_weights[:num_tokens] = query_weights
    
    try:
        all_page_data = {} # extraction of all patch embeddings per page 
        with conn.cursor() as cur:
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        scored_pages = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32) # np matrix for page
            sim_matrix = query_embeddings @ page_matrix.T # matrix multiplication 
            token_scores = np.max(sim_matrix, axis=1) # sum best matching patches for page
            
            scored_pages.append({
                "id": p_id,
                "token_scores": token_scores * final_weights
            })

        # diversity selection (marginal gain logic)
        selected_pages = []
        best_scores_per_token = np.zeros(query_embeddings.shape[0]) # running coverage of query tokens

        for _ in range(min(top_k, len(scored_pages))):
            best_marginal_gain = -1.0
            best_candidate = None
            
            for page in scored_pages:
                if page in selected_pages:
                    continue
                
                gains = np.maximum(0, page["token_scores"] - best_scores_per_token) # marginal gain
                marginal_gain = np.sum(gains)
                
                if marginal_gain > best_marginal_gain:
                    best_marginal_gain = marginal_gain
                    best_candidate = page
            
            if best_candidate:
                selected_pages.append(best_candidate)
                best_scores_per_token = np.maximum(best_scores_per_token, best_candidate["token_scores"]) # update running coverage
        
        return [(p["id"], np.sum(p["token_scores"])) for p in selected_pages], scored_pages

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []



In [203]:
def precision_search_diverse(query_embeddings, ids, top_k=10):
    
    try:
        all_page_data = {} # extraction of all patch embeddings per page 
        with conn.cursor() as cur:
            cur.execute('''
                SELECT page_id, patch_embedding 
                FROM page_patches 
                WHERE page_id IN %s 
                ORDER BY page_id, patch_index
            ''', (tuple(ids),))
            
            rows = cur.fetchall()
            for p_id, emb in rows:
                if p_id not in all_page_data:
                    all_page_data[p_id] = []
                all_page_data[p_id].append(emb)

        scored_pages = []
        for p_id, patches in all_page_data.items():
            page_matrix = np.array(patches, dtype=np.float32) # np matrix for page
            sim_matrix = query_embeddings @ page_matrix.T # matrix multiplication 
            token_scores = np.max(sim_matrix, axis=1) # sum best matching patches for page
            
            scored_pages.append({
                "id": p_id,
                "token_scores": token_scores
            })

        # diversity selection (marginal gain logic)
        selected_pages = []
        best_scores_per_token = np.zeros(query_embeddings.shape[0]) # running coverage of query tokens

        for _ in range(min(top_k, len(scored_pages))):
            best_marginal_gain = -1.0
            best_candidate = None
            
            for page in scored_pages:
                if page in selected_pages:
                    continue
                
                gains = np.maximum(0, page["token_scores"] - best_scores_per_token) # marginal gain
                marginal_gain = np.sum(gains)
                
                if marginal_gain > best_marginal_gain:
                    best_marginal_gain = marginal_gain
                    best_candidate = page
            
            if best_candidate:
                selected_pages.append(best_candidate)
                best_scores_per_token = np.maximum(best_scores_per_token, best_candidate["token_scores"]) # update running coverage
        
        return [(p["id"], np.sum(p["token_scores"])) for p in selected_pages], scored_pages

    except Exception as e:
        print(f"Database Error: {e}")
        conn.rollback()
        return []


In [17]:
# MaxSim
def precision_search(embeddings, ids, top_k = 10):
    try: 
        result = []
        for p_id in ids:
            with conn.cursor() as cur:
                cur.execute('''
                    SELECT patch_embedding
                    FROM page_patches
                    WHERE page_id = %s
                    ORDER BY patch_index
                ''', (p_id,))
            
                rows = cur.fetchall()
                page_matrix = np.array([r[0] for r in rows], dtype = np.float32) # np matrix for page
                sim_matrix = embeddings @ page_matrix.T # matrix multiplication
                page_score = np.sum(np.max(sim_matrix, axis=1)) # sum best matching patches for page
                result.append((p_id, page_score))
        return sorted(result, key=lambda x: x[1], reverse=True)[:top_k]

    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [18]:
def retrieve_content(results):
    output = []
    with conn.cursor() as cur:
        for id, score in results:
            cur.execute('''
                SELECT page_number, markdown
                FROM pages
                WHERE id = %s
            ''', (id,))

            row = cur.fetchone()
            output.append({
                "page_number": row[0],
                "markdown": row[1]
            })
    return output

In [19]:
def get_page_base64(pdf_path, page_number):
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)

    zoom = 1.0 
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)

    img_bytes = pix.tobytes("jpg")
    return base64.b64encode(img_bytes).decode('utf-8')


In [20]:
def generate_context(content, pdf_path):
    context = []
    for dic in content:
        context.append({
            "type": "text",
            "text": f"--- Page {dic['page_number']} ---\n{dic['markdown']}"
        })

        b64_img = get_page_base64(pdf_path, dic["page_number"])

        context.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpg;base64,{b64_img}"}
        })
    return HumanMessage(content=context)

In [21]:
def generate_message(query, context):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context which includes the markdown and image of pages of a document.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. DO NOT USE EXTERNAL KNOWLEDGE.
    2. MARKDOWN: Use the markdown text for precise data extractions, quotes and tables.
    3. IMAGES: Use the images to verify layout, check for visual elements, and clarify ambiguous parts of the text.
    4. DISCREPANCY: Treat the page image and markdown as equal independent sources of information. ANY CLAIM OR INFORMATION MUST BE SUPPORTED BY BOTH SOURCES. Flag any discrepancies. 
    5. CITATIONS: You MUST CITE THE SPECIFIC PAGE NUMBER (e.g. [Page 4]) for every fact or claim you make. Each page is labelled at the start (e.g. --- Page 4 ---).
    6. CONTEXT INTEGRATION: Treat the context as a single unified knowledge base, even if they are provided out of chronological order.
    7. RELEVANCE: ONLY USE INFORMATION FROM THE CONTEXT that are relevant to answering the question.  
    8. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    context, 
    HumanMessage(content=f"""
    Question: {query}
    """)

    ]
    return messages

TESTING

In [329]:
query = "What types of goods and services does Tender Lite currently cover, and will it expand?"

In [336]:
import time

t0 = time.perf_counter()
coarse_vector, embeddings = embed_query(query, model, processor)
ids = coarse_search(coarse_vector)
results = precision_search_diverse_weighted(query, embeddings, ids)
print(f"Search done at: {(time.perf_counter() - t0)*1000:.2f}ms")
content = retrieve_content(results[0])
print(f"Content retrieval done at: {(time.perf_counter() - t0)*1000:.2f}ms")
context = generate_context(content, pdf_path)
message = generate_message(query, context)
print(f"Message preparation done at: {(time.perf_counter() - t0)*1000:.2f}ms")
first_token_received = False

for chunk in llm.stream(message):
    if not first_token_received:
        ttft = (time.perf_counter() - t0) * 1000
        print(f"\n[STREAM START] Time to first token: {ttft:.2f}ms")
        first_token_received = True
    
    print(chunk.content, end="", flush=True)

print(f"\n\nTotal time to completion: {(time.perf_counter() - t0)*1000:.2f}ms")

Search done at: 1290.60ms
Content retrieval done at: 1293.57ms
Message preparation done at: 1366.81ms

[STREAM START] Time to first token: 4150.47ms
### Tender Lite Coverage

- **Current Coverage**: Tender Lite is a sub-category of Tender that covers new tender opportunities for general goods and services with an estimated procurement value not exceeding S$1 million. 

- **Future Expansion**: The Ministry of Finance (MOF) will be reviewing Tender Lite for Information and Communication Technology (ICT) and construction purchases in the following years, indicating potential expansion into these areas [Page 7].

Total time to completion: 5188.50ms


In [338]:
content

[{'page_number': 7,
  'markdown': "## [New] Tender Lite\n\nTender Lite is a sub-category of Tender with estimated procurement value not exceeding S$1 million. It will cover new tender opportunities called for general goods and services from end April 2024 . MOF will be reviewing Tender Lite for ICT and construction buys in following years. Under Tender Lite, there would be fewer contract conditions as compared to typical tenders. For more information on how to search and participate in Tender Lite opportunities, please find user guide at https://www.gebiz.gov.sg/docs/Supplier\\_Guide\\_Tender\\_Lite.pdf .\n\n## Demand Aggregation (DA)\n\nAgencies  may  also  combine  purchases  of  common  goods and  services  by  establishing  a  Demand Aggregation contract to yield better value for money through economies of scale. These contracts may be called by an agency for their own use or on behalf of other agencies, and are usually established through  an  open  Tender  or  Quotation  process.

In [104]:
def search_page_no(page_id):
    try: 
        with conn.cursor() as cur:
            cur.execute("SELECT page_number FROM PAGES WHERE id = %s", (page_id,))
            page_no = cur.fetchone()
        return page_no
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()



In [ ]:
scored_pages = results[1]
inputs = processor(text=[query], return_tensors="pt")
input_ids = inputs["input_ids"][0]
query_tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
clean_tokens = [t.replace('Ġ', ' ').replace('Ċ', '\n') for t in query_tokens]
for page in scored_pages:
    p_no = search_page_no(page["id"])[0]
    scores = page["token_scores"]
    
    print(f"\n{'='*30}")
    print(f" ANALYSIS FOR PAGE NO: {p_no}")
    print(f"{'='*30}")
    
    for token, score in zip(clean_tokens, scores):
        print(f"{token:15} | Score: {score:.4f}")
